# Hand-Labeling Calcium Events with `EventLabeler`

This notebook walks through using `wizards_staff.labeling.EventLabeler` to
build a hand-labeled corpus of calcium events for downstream threshold
calibration and (eventually) classifier training.

## What you will do

1. Load a `Shard` produced by a Wizards-Staff `run_all` invocation.
2. Instantiate an `EventLabeler` and tag your session with experiment
   context (sampling rate, indicator, microscope, cell type, experiment
   id) so that every label you produce is stratifiable later.
3. Call `.display()` and click through the events.
4. Reload the canonical CSV corpus and inspect what you (and other
   labelers) have contributed.

## Setup

The interactive UI requires `ipywidgets`. The rest of `wizards_staff`
does not — `ipywidgets` is gated behind the `labeling` extra so that
headless callers (e.g. Lizard-Wizard's CLI) do not need a Jupyter widget
stack:

```bash
pip install 'wizards_staff[labeling]'
```

Run that once on the JupyterHub kernel you'll be labeling from.

In [ ]:
%matplotlib inline

import os
from pathlib import Path

import pandas as pd

from wizards_staff.wizards.orb import Orb
from wizards_staff.labeling import EventLabeler

## 1. Load a shard from a real `run_all` output

Point `RESULTS_FOLDER` at the directory that contains your
Lizard-Wizard outputs and `METADATA_FILE` at the matching
`Sample,Well,Frate` CSV, then pick the sample you want to label.

The `run_all` call is what actually populates `_raw_peak_amplitude_data`
and `_raw_fwhm_data` on each shard — the labeler walks those raw lists,
so the `filter_events=False` setting below is irrelevant to the labeling
tool itself, but is a reasonable default for calibration runs because
you usually want to see every detected event.

In [ ]:
RESULTS_FOLDER = "/path/to/lizard_wizard/output"
METADATA_FILE = "/path/to/metadata.csv"
SAMPLE_NAME = "my_sample_id"

orb = Orb(
    results_folder=RESULTS_FOLDER,
    metadata_file_path=METADATA_FILE,
)
orb.run_all(
    frate=30,
    filter_events=False,
)

shard = orb._shards[SAMPLE_NAME]
print(
    f"Loaded {SAMPLE_NAME}: "
    f"{len(shard._raw_peak_amplitude_data)} neurons, "
    f"{sum(len(r['Peak Amplitudes']) for r in shard._raw_peak_amplitude_data)} raw events"
)

## 2. Configure the labeling session

The corpus is a single CSV that grows across sessions and labelers. Put
it on shared storage so multiple biologists can contribute to the same
training set; the labeler writes atomically and uses last-write-wins for
concurrent writes (documented; concurrent multi-labeler support is
future work).

Everything you put in `context` is stored on every row, which is what
makes the corpus stratifiable later (e.g. "give me only the GCaMP6f
events at 30 Hz on the spinning disk scope").

In [ ]:
CORPUS_PATH = "/shared/wizards_staff/event_labels.csv"
LABELER_ID = os.environ.get("USER", "unknown_labeler")

labeler = EventLabeler(
    shard=shard,
    corpus_path=CORPUS_PATH,
    labeler_id=LABELER_ID,
    context={
        "sampling_rate": 30,
        "indicator": "GCaMP6f",
        "microscope": "spinning_disk",
        "cell_type": "cortical_neuron",
        "experiment_id": "exp_2026_04_chronic_stim",
    },
    window_scale=8.0,
    ordering="by_roi_then_time",
)
print(f"{labeler.total_events} events queued; {labeler.labeled_count} already labeled by you.")

## 3. Label

Run the cell below to render the UI. Each event is shown with:

- A zoomed trace window centred on the event, with the FWHM region
  shaded and the peak marked.
- A full-trace minimap so you can see the cell's overall behaviour.
- Tick marks showing previously-labeled events on the same neuron
  (green = True, red = False, gray = Unsure).

Workflow:

- Click `True`, `False`, or `Unsure` (or click the `Keys:` field and
  press `t`, `f`, `u`).
- Use `j` / `k` (or the Prev / Next buttons) to move without labeling.
- If every detected event on a neuron's trace is clearly noise, hit `Reject whole trace` (`w`). This labels the events False; it does not mark the neuron itself bad (whole-neuron rejection lives in the outlier-detection layer).
- Optional notes go into the `Notes:` field before you label.
- Every action is persisted atomically. You can close the notebook at
  any time and resume later by re-running the cells above.

The progress bar at the top shows where you are within the current
neuron, within the shard, and across all events labeled so far.

In [ ]:
labeler.display()

## 4. Inspect the corpus

Use `EventLabeler.load_corpus(...)` to read the CSV with the dtypes the
tool intended (in particular it forces `label` to `str`, which prevents
`pandas.read_csv` from silently inferring `bool` when a calibration run
happens to contain only `True`/`False` rows).

From here you can stratify by indicator/microscope/etc. for threshold
calibration, or hand the file off to a downstream classifier-training
step.

In [ ]:
corpus = EventLabeler.load_corpus(CORPUS_PATH)
print(f"Corpus rows: {len(corpus)}")
print(f"Labelers contributing: {sorted(corpus['labeler_id'].dropna().unique())}")
print(f"Samples covered:      {sorted(corpus['sample_id'].dropna().unique())}")
print("Label distribution overall:")
print(corpus['label'].value_counts())
corpus.head()

In [ ]:
by_indicator = (
    corpus
    .groupby(['indicator', 'label'])
    .size()
    .unstack(fill_value=0)
)
by_indicator

In [ ]:
session = labeler.export_labels()
print(f"This session contributed {len(session)} labels for {SAMPLE_NAME}.")
session[['roi_id', 'event_idx', 'label', 'peak_amplitude', 'fwhm_frames', 'notes']].head(20)